In [1]:
import os
import time
import threading
import wave
from pathlib import Path

import numpy as np
import sounddevice as sd
import soundfile as sf
import whisper
import pvporcupine
from gtts import gTTS
import cohere


# === DIRECTORIES & FILE PATHS =============================
AUDIO_RECORD_PATH = r"C:\Users\MichelleGh\Desktop\Graduation Project\Verbal Communication\6-V_COM\1-AUDIO_RECORD_PATH\recorded_input.wav"
STT_OUTPUT_PATH = r"C:\Users\MichelleGh\Desktop\Graduation Project\Verbal Communication\6-V_COM\2-STT_OUPUT_PATH\STT_Output.txt"
AI_REPLY_PATH = r"C:\Users\MichelleGh\Desktop\Graduation Project\Verbal Communication\6-V_COM\3-AI_REPLY_PATH\ai_reply.txt"
TTS_OUTPUT_PATH = r"C:\Users\MichelleGh\Desktop\Graduation Project\Verbal Communication\6-V_COM\4-TTS_OUTPUT_PATH\Robot_Response.wav"

YES_PATH = r"C:\Users\MichelleGh\Desktop\Graduation Project\Verbal Communication\6-V_COM\5-YES_PATH\yes-masterrr.wav"
WAKEWORD_MODEL = r"C:\Users\MichelleGh\Desktop\Graduation Project\Verbal Communication\6-V_COM\6-WAKEWORD_MODEL\Hey-Robot_en_windows_v3_0_0.ppn"


# === DEVICE SELECTION ===
INPUT_DEVICE_ID = 8
OUTPUT_DEVICE_ID = 10
sd.default.device = (INPUT_DEVICE_ID, OUTPUT_DEVICE_ID)


# === API KEYS ===
ACCESS_KEY = "1UQRnlg6WE6MbpJO1jX135lftC3GCuJE8+sL7hIVqtiQwLfkfjfrhg=="
COHERE_API_KEY = "pxbp77BALEELCHPMtk6IGPv3iejrSUpwYFGWm5ML"


# === INITIALIZE MODELS ===
print("Loading Whisper model...")
whisper_model = whisper.load_model("base")

print("Loading 'YES' audio...")
YES_audio, YES_sr = sf.read(YES_PATH, dtype='float32')

print("Initializing Cohere...")
co = cohere.Client(api_key=COHERE_API_KEY)

print("Loading wake word model...")
porcupine = pvporcupine.create(
    access_key=ACCESS_KEY,
    keyword_paths=[WAKEWORD_MODEL]
)
FRAME_LENGTH = porcupine.frame_length



# === HELPER FUNCTIONS ====
def play_yes():
    sd.play(YES_audio, YES_sr, device=OUTPUT_DEVICE_ID)
    sd.wait()


def auto_record_voice():
    """Immediately record after wake-word and stop on silence."""
    print("\nRecording... (speak now)")

    frames = []
    silence_threshold = 0.01
    silence_duration = 2.0  # Seconds of silence to stop recording
    silence_time = 0.0

    device_info_input = sd.query_devices(INPUT_DEVICE_ID)
    sample_rate = int(device_info_input['default_samplerate'])
    blocksize = 1024

    def rms(data):
        return np.sqrt(np.mean(np.square(data)))

    def callback(indata, frames_count, time_info, status):
        nonlocal silence_time
        frames.append(indata.copy())
        if rms(indata) < silence_threshold:
            silence_time += blocksize / sample_rate
        else:
            silence_time = 0.0

    stream = sd.InputStream(
        samplerate=sample_rate,
        channels=1,
        callback=callback,
        blocksize=blocksize,
        device=INPUT_DEVICE_ID
    )

    stream.start()

    while silence_time < silence_duration:
        time.sleep(0.05)

    stream.stop()
    stream.close()

    print("Silence detected — stopping recording.")

    recording = np.concatenate(frames, axis=0)

    with wave.open(AUDIO_RECORD_PATH, 'wb') as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sample_rate)
        wf.writeframes((recording * 32767).astype(np.int16).tobytes())

    print(f"Saved recording to: {AUDIO_RECORD_PATH}")


def speech_to_text():
    print("\nTranscribing with Whisper...")
    result = whisper_model.transcribe(AUDIO_RECORD_PATH)
    text = result['text']

    Path(STT_OUTPUT_PATH).write_text(text, encoding="utf-8")
    print(f"STT saved to: {STT_OUTPUT_PATH}")
    return text


def ask_ai(text):
    print("\nSending text to Cohere...")
    response = co.chat(model="command-a-03-2025", message=text)
    ai_text = response.text

    Path(AI_REPLY_PATH).write_text(ai_text, encoding="utf-8")
    print(f"AI reply saved to: {AI_REPLY_PATH}")
    return ai_text


def text_to_speech(text):
    print("\nGenerating speech using gTTS...")
    tts = gTTS(text)
    tts.save(TTS_OUTPUT_PATH)
    print(f"TTS saved to: {TTS_OUTPUT_PATH}")


def play_tts():
    """Play the final TTS audio."""
    print("Playing TTS audio... Hold on.")
    if Path(TTS_OUTPUT_PATH).exists():
        audio, sr = sf.read(TTS_OUTPUT_PATH, dtype='float32')
        sd.play(audio, sr, device=OUTPUT_DEVICE_ID)
        sd.wait()


# === MAIN SYSTEM LOOP ===
def main():
    print("\n=== SYSTEM READY — Say 'Hey Robot' ===\n")

    def audio_callback(indata, frames, time_info, status):
        pcm = (indata[:, 0] * 32767).astype("int16")
        keyword_index = porcupine.process(pcm)

        if keyword_index >= 0:
            print("\nWake word detected!")
            threading.Thread(target=play_yes, daemon=True).start()

            # === FULL PIPELINE ===
            auto_record_voice()
            text = speech_to_text()
            ai_reply = ask_ai(text)
            text_to_speech(ai_reply)
            play_tts()

            print("\nSystem ready again — say 'Hey Robot'\n")

    with sd.InputStream(
        device=INPUT_DEVICE_ID,
        channels=1,
        samplerate=porcupine.sample_rate,
        blocksize=FRAME_LENGTH,
        dtype='float32',
        callback=audio_callback
    ):
        try:
            while True:
                time.sleep(0.1)
        except KeyboardInterrupt:
            print("Exiting...")
        finally:
            porcupine.delete()


if __name__ == "__main__":
    main()


Loading Whisper model...
Loading 'YES' audio...
Initializing Cohere...
Loading wake word model...

=== SYSTEM READY — Say 'Hey Robot' ===


Wake word detected!

Recording... (speak now)
Silence detected — stopping recording.
Saved recording to: C:\Users\MichelleGh\Desktop\Graduation Project\Verbal Communication\6-V_COM\1-AUDIO_RECORD_PATH\recorded_input.wav

Transcribing with Whisper...


c:\Users\MichelleGh\AppData\Local\Programs\Python\Python313\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


STT saved to: C:\Users\MichelleGh\Desktop\Graduation Project\Verbal Communication\6-V_COM\2-STT_OUPUT_PATH\STT_Output.txt

Sending text to Cohere...
AI reply saved to: C:\Users\MichelleGh\Desktop\Graduation Project\Verbal Communication\6-V_COM\3-AI_REPLY_PATH\ai_reply.txt

Generating speech using gTTS...
TTS saved to: C:\Users\MichelleGh\Desktop\Graduation Project\Verbal Communication\6-V_COM\4-TTS_OUTPUT_PATH\Robot_Response.wav
Playing TTS audio... Hold on.

System ready again — say 'Hey Robot'

Exiting...
